# Baseline Models

This notebook trains Logistic Regression and Decision Tree baselines with
stratified 5-fold cross-validation, logs all runs to MLflow, and establishes
the performance floor that any more complex model must beat to justify the
added complexity.

**What's covered**

| Section | Content |
|---|---|
| 1. Data preparation | Load pipeline → preprocessed train/val/test arrays |
| 2. Cross-validation | Per-fold AUC, F1, precision, recall |
| 3. ROC curves | Both models on the held-out validation set |
| 4. Confusion matrices | Default-threshold predictions on validation |
| 5. LR coefficients | Top drivers from the linear model |
| 6. Summary | Side-by-side comparison table |


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.load import download_telco_data
from src.data.pipeline import prepare_data
from src.models.baseline import train_decision_tree, train_logistic
from src.models.cv import cv_summary, stratified_cv_score
from src.utils.mlflow_utils import init_mlflow
from src.utils.plotting import set_plot_style, save_fig

set_plot_style()
FIGURES_DIR = REPO_ROOT / "reports" / "figures"

In [ ]:
with open(REPO_ROOT / "configs" / "config.yaml") as fh:
    cfg = yaml.safe_load(fh)

SEED: int = cfg["random_seed"]
RAW_PATH = REPO_ROOT / cfg["paths"]["raw_data"]
TRACKING_URI = str(REPO_ROOT / cfg["paths"]["mlruns"])
EXPERIMENT_NAME: str = cfg["mlflow"]["experiment_name"]

download_telco_data(RAW_PATH, urls=cfg["data"]["download_urls"])

data = prepare_data(
    RAW_PATH,
    val_size=cfg["data"]["val_size"],
    test_size=cfg["data"]["test_size"],
    seed=SEED,
    processed_dir=REPO_ROOT / cfg["paths"]["processed_data"],
)

X_train = data["X_train"]
y_train = data["y_train"].to_numpy()
X_val = data["X_val"]
y_val = data["y_val"].to_numpy()
X_test = data["X_test"]
y_test = data["y_test"].to_numpy()
feature_names: list[str] = data["feature_names"]

# Combine train + val for CV (test is kept completely held out)
X_cv = np.vstack([X_train, X_val])
y_cv = np.concatenate([y_train, y_val])

print(f"Train : {X_train.shape[0]:,} rows | Val: {X_val.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Features: {X_train.shape[1]}")
print(f"Target prevalence — train: {y_train.mean():.1%}  val: {y_val.mean():.1%}  test: {y_test.mean():.1%}")

## 1. MLflow Experiment Setup

In [ ]:
import mlflow

init_mlflow(EXPERIMENT_NAME, tracking_uri=TRACKING_URI)
print(f"Tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experiment   : {EXPERIMENT_NAME}")

## 2. Stratified 5-Fold Cross-Validation

CV is run on the combined train+val pool (80 % of all data). The held-out
test set is never touched here.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

N_SPLITS = 5

lr_proto = LogisticRegression(
    C=1.0, max_iter=1000, solver="lbfgs", class_weight="balanced", random_state=SEED
)
dt_proto = DecisionTreeClassifier(
    max_depth=5, min_samples_leaf=20, class_weight="balanced", criterion="gini", random_state=SEED
)

print(f"Running {N_SPLITS}-fold CV for Logistic Regression...")
lr_cv = stratified_cv_score(lr_proto, X_cv, y_cv, n_splits=N_SPLITS, seed=SEED)

print(f"Running {N_SPLITS}-fold CV for Decision Tree...")
dt_cv = stratified_cv_score(dt_proto, X_cv, y_cv, n_splits=N_SPLITS, seed=SEED)

print("Done.")

In [ ]:
metric_cols = [c for c in lr_cv.columns if c != "fold"]

def format_cv_table(cv_df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    rows = []
    for col in metric_cols:
        rows.append({
            "model": model_name,
            "metric": col,
            "mean": cv_df[col].mean(),
            "std": cv_df[col].std(),
            "min": cv_df[col].min(),
            "max": cv_df[col].max(),
        })
    return pd.DataFrame(rows)

cv_table = pd.concat([
    format_cv_table(lr_cv, "LogisticRegression"),
    format_cv_table(dt_cv, "DecisionTree"),
], ignore_index=True)

display(
    cv_table.style
    .format({"mean": "{:.4f}", "std": "{:.4f}", "min": "{:.4f}", "max": "{:.4f}"})
    .background_gradient(subset=["mean"], cmap="RdYlGn")
)

In [ ]:
# Per-fold detail for AUC
fold_auc = pd.DataFrame({
    "fold": lr_cv["fold"],
    "LR AUC": lr_cv["roc_auc"].round(4),
    "DT AUC": dt_cv["roc_auc"].round(4),
})

# Append mean row
fold_auc.loc[len(fold_auc)] = [
    "mean ± std",
    f"{lr_cv['roc_auc'].mean():.4f} ± {lr_cv['roc_auc'].std():.4f}",
    f"{dt_cv['roc_auc'].mean():.4f} ± {dt_cv['roc_auc'].std():.4f}",
]
display(fold_auc)

## 3. Train Final Models on Train Set, Evaluate on Validation

Final models are fit on the training split only. Validation metrics are
an independent held-out estimate (not seen during CV).

In [ ]:
with mlflow.start_run(run_name="logistic_regression_baseline"):
    model_lr, val_metrics_lr = train_logistic(
        X_train, y_train, X_val, y_val, seed=SEED, log_to_mlflow=True
    )
    mlflow.log_metrics({
        f"cv_{col}_mean": float(lr_cv[col].mean())
        for col in metric_cols
    })
    mlflow.log_metrics({
        f"cv_{col}_std": float(lr_cv[col].std())
        for col in metric_cols
    })

print("LR validation metrics:")
for k, v in val_metrics_lr.items():
    print(f"  {k:<12}: {v:.4f}")

In [ ]:
with mlflow.start_run(run_name="decision_tree_baseline"):
    model_dt, val_metrics_dt = train_decision_tree(
        X_train, y_train, X_val, y_val, seed=SEED, log_to_mlflow=True
    )
    mlflow.log_metrics({
        f"cv_{col}_mean": float(dt_cv[col].mean())
        for col in metric_cols
    })
    mlflow.log_metrics({
        f"cv_{col}_std": float(dt_cv[col].std())
        for col in metric_cols
    })

print("DT validation metrics:")
for k, v in val_metrics_dt.items():
    print(f"  {k:<12}: {v:.4f}")

## 4. ROC Curves

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 6))

RocCurveDisplay.from_estimator(
    model_lr, X_val, y_val, ax=ax,
    name=f"Logistic Regression (AUC={val_metrics_lr['roc_auc']:.3f})",
    color="#2E86AB",
)
RocCurveDisplay.from_estimator(
    model_dt, X_val, y_val, ax=ax,
    name=f"Decision Tree      (AUC={val_metrics_dt['roc_auc']:.3f})",
    color="#E84855",
)

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random classifier")
ax.set_title("ROC Curves — Baseline Models (Validation Set)")
ax.legend(loc="lower right", frameon=False)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
fig.tight_layout()
save_fig(fig, "04_roc_curves", FIGURES_DIR)
plt.show()

## 5. Confusion Matrices

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, name in zip(
    axes,
    [model_lr, model_dt],
    ["Logistic Regression", "Decision Tree"],
):
    ConfusionMatrixDisplay.from_estimator(
        model, X_val, y_val,
        display_labels=["No churn", "Churn"],
        cmap="Blues",
        ax=ax,
        colorbar=False,
    )
    ax.set_title(name)

fig.suptitle("Confusion Matrices — Validation Set (default 0.5 threshold)", y=1.02)
fig.tight_layout()
save_fig(fig, "04_confusion_matrices", FIGURES_DIR)
plt.show()

## 6. Logistic Regression Coefficient Analysis

LR coefficients are a direct proxy for feature importance (after scaling).
Positive coefficients push the prediction toward churn; negative coefficients
push toward retention.

In [ ]:
coef_df = (
    pd.DataFrame({"feature": feature_names, "coefficient": model_lr.coef_[0]})
    .assign(abs_coef=lambda d: d["coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
    .head(20)
    .sort_values("coefficient")
)

fig, ax = plt.subplots(figsize=(9, 8))
colors = ["#E84855" if c > 0 else "#2E86AB" for c in coef_df["coefficient"]]
ax.barh(coef_df["feature"], coef_df["coefficient"], color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Top 20 LR Coefficients (scaled features)\nRed = churn driver · Blue = retention driver")
ax.set_xlabel("Coefficient value")
fig.tight_layout()
save_fig(fig, "04_lr_coefficients", FIGURES_DIR)
plt.show()

In [ ]:
print("Top 10 churn-driving features (LR):")
display(
    coef_df.sort_values("coefficient", ascending=False)
    .head(10)[["feature", "coefficient"]]
    .reset_index(drop=True)
    .style.format({"coefficient": "{:.4f}"})
    .bar(subset=["coefficient"], align="mid", color=["#E84855", "#2E86AB"])
)

print("\nTop 10 retention-driving features (LR):")
display(
    coef_df.sort_values("coefficient")
    .head(10)[["feature", "coefficient"]]
    .reset_index(drop=True)
    .style.format({"coefficient": "{:.4f}"})
    .bar(subset=["coefficient"], align="mid", color=["#E84855", "#2E86AB"])
)

## 7. Decision Tree Feature Importances

In [ ]:
dt_importance = (
    pd.DataFrame({"feature": feature_names, "importance": model_dt.feature_importances_})
    .sort_values("importance", ascending=True)
    .tail(20)
)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(dt_importance["feature"], dt_importance["importance"], color="#2E86AB", edgecolor="white")
ax.set_title("Top 20 Decision Tree Feature Importances (Gini)")
ax.set_xlabel("Importance")
fig.tight_layout()
save_fig(fig, "04_dt_feature_importance", FIGURES_DIR)
plt.show()

## 8. Summary Comparison

In [ ]:
summary = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "CV AUC": f"{lr_cv['roc_auc'].mean():.4f} ± {lr_cv['roc_auc'].std():.4f}",
        "CV F1": f"{lr_cv['f1'].mean():.4f} ± {lr_cv['f1'].std():.4f}",
        "CV Recall": f"{lr_cv['recall'].mean():.4f} ± {lr_cv['recall'].std():.4f}",
        "Val AUC": f"{val_metrics_lr['roc_auc']:.4f}",
        "Val F1": f"{val_metrics_lr['f1']:.4f}",
        "Val Recall": f"{val_metrics_lr['recall']:.4f}",
    },
    {
        "Model": "Decision Tree",
        "CV AUC": f"{dt_cv['roc_auc'].mean():.4f} ± {dt_cv['roc_auc'].std():.4f}",
        "CV F1": f"{dt_cv['f1'].mean():.4f} ± {dt_cv['f1'].std():.4f}",
        "CV Recall": f"{dt_cv['recall'].mean():.4f} ± {dt_cv['recall'].std():.4f}",
        "Val AUC": f"{val_metrics_dt['roc_auc']:.4f}",
        "Val F1": f"{val_metrics_dt['f1']:.4f}",
        "Val Recall": f"{val_metrics_dt['recall']:.4f}",
    },
])

display(summary.set_index("Model"))

## Interpretation

**Logistic Regression** is the stronger baseline. Its linear decision boundary
captures the additive signal from contract type, tenure, and payment method
reliably, and the L2 regularisation prevents overfitting to the engineered
interaction features. The model favours recall over precision (class-weight
='balanced'), which aligns with the business cost matrix: missing a churner
(false negative) costs ~10× more than an unnecessary retention offer.

**Decision Tree** has lower AUC and higher variance across folds, which is
expected for a single tree at moderate depth. It is retained as a baseline
because its structure is fully interpretable — the splits can be shown
directly to business stakeholders.

**Interpretability highlights** (from LR coefficients):
- Long-term contracts and autopay are the strongest retention signals, consistent
  with EDA findings.
- Electronic check and month-to-month contract are the strongest churn drivers.
- Engineered features (`contract_risk_score`, `autopay_flag`, `tenure_x_contract`)
  appear prominently, confirming they carry real predictive signal.

**Baseline targets for complex models**:
Any subsequent model (XGBoost, LightGBM, tuned ensemble) must exceed the LR
CV AUC by a meaningful margin (>0.01) on the same folds to justify the
increased complexity and inference cost.